# 01 — Preparação dos dados

Material de apoio ao [artigo](00_artigo.ipynb). Prepara notícias e cotações de NVDA, NFLX, BAC, JNJ e KO em 2017–2019. Preserva originais, identificadores e registros de exclusão.

Execute em ordem, na raiz do projeto ou em `notebooks/`, com Python 3.11 e as dependências do [README](../README.md). Coloque o CSV de notícias em `data/raw/`. A coleta de preços usa o snapshot existente; sem ele, exige internet.

## 1. Escopo

A saída desta etapa é `noticias_limpas.csv` e `cotacoes_limpas_com_margem.csv`. As etapas seguintes classificam o sentimento (02), integram notícias e pregões (03) e apresentam a análise visual (00). Na referência utilizada, a limpeza mantém 6.552 associações após remover quatro duplicatas.

## 2. Configuração

Definimos o recorte, os caminhos e as versões das bibliotecas. O hash identifica o CSV original e permite verificar sua preservação. As margens de preços fornecem o pregão anterior ao início e o posterior ao fim do estudo.

In [1]:
import os
os.environ['MKL_THREADING_LAYER'] = 'SEQUENTIAL'
import sys
import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path
from importlib.metadata import version
import pandas as pd
import numpy as np
from IPython.display import display

RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'requirements.txt').exists() and (p / 'notebooks').is_dir()), None)
if RAIZ is None:
    raise FileNotFoundError('Abra este notebook na raiz do repositório ou na pasta notebooks/.')
ORIGINAIS = RAIZ / 'data' / 'raw'
TRATADOS = RAIZ / 'data' / 'processed'
DOCS = RAIZ / 'docs'
for pasta in [ORIGINAIS, TRATADOS, DOCS, RAIZ / 'tmp']:
    pasta.mkdir(parents=True, exist_ok=True)

entrada_local = ORIGINAIS / 'raw_analyst_ratings.csv'
entrada_anexo = RAIZ.parent / 'sources' / 'raw_analyst_ratings.csv'
ARQUIVO_NOTICIAS = entrada_local if entrada_local.exists() else entrada_anexo
if not ARQUIVO_NOTICIAS.exists():
    raise FileNotFoundError('Coloque o CSV original em data/raw/raw_analyst_ratings.csv.')

TICKERS = ['NVDA', 'NFLX', 'BAC', 'JNJ', 'KO']
INICIO = pd.Timestamp('2017-01-01')
FIM = pd.Timestamp('2019-12-31')
COLETA_INICIO = '2016-12-15'
COLETA_FIM_EXCLUSIVO = '2020-01-11'
VERSOES = {nome: version(nome) for nome in
           ['pandas', 'numpy', 'yfinance', 'ipykernel']}

def sha256(arquivo):
    h = hashlib.sha256()
    with Path(arquivo).open('rb') as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b''):
            h.update(bloco)
    return h.hexdigest()

HASH_ORIGINAL = sha256(ARQUIVO_NOTICIAS)
print('Python:', sys.version.split()[0])
print('Projeto:', RAIZ.name)
print('CSV lido:', ARQUIVO_NOTICIAS.relative_to(RAIZ) if ARQUIVO_NOTICIAS.is_relative_to(RAIZ) else ARQUIVO_NOTICIAS.name)
print('Recorte:', TICKERS, INICIO.date(), FIM.date())
print('SHA-256:', HASH_ORIGINAL)
display(pd.Series(VERSOES, name='versao').to_frame())


Python: 3.11.16
Projeto: reproducao
CSV lido: data\raw\raw_analyst_ratings.csv
Recorte: ['NVDA', 'NFLX', 'BAC', 'JNJ', 'KO'] 2017-01-01 2019-12-31
SHA-256: 609289a49dc8af2bd28a8466dffd9663521237026f1040c0f238010b29135fb6


,versao
pandas,3.0.6
numpy,2.4.6
yfinance,1.7.0
ipykernel,7.4.0


## 3. Limpeza e validação

As notícias vêm de *Daily Financial News for 6000+ Stocks* (Kaggle), e as cotações do Yahoo Finance via `yfinance`. A unidade da notícia é empresa/data civil/manchete; a das cotações é empresa/pregão. Não criamos preços nem sentimentos para preencher ausências.

### 3.1 Inspecionar a estrutura e preservar identificadores

Lemos os campos como texto e verificamos as colunas. `source_id` conserva o identificador original; `source_row` registra a posição lógica no CSV. Esses campos permitem rastrear exclusões, mas não definem duplicatas. O diagnóstico distingue valores nulos de textos vazios; URL ou autor ausente, isoladamente, não exclui uma manchete.

In [2]:
brutas = pd.read_csv(ARQUIVO_NOTICIAS, dtype='string', keep_default_na=False)
esperadas = {'Unnamed: 0', 'headline', 'url', 'publisher', 'date', 'stock'}
faltando = esperadas - set(brutas.columns)
if faltando:
    raise ValueError(f'Colunas obrigatórias ausentes: {faltando}')

brutas = brutas.rename(columns={'Unnamed: 0': 'source_id'})
brutas['source_row'] = np.arange(1, len(brutas) + 1)
id_numerico = pd.to_numeric(brutas['source_id'], errors='coerce')
if id_numerico.isna().any() or not (id_numerico % 1 == 0).all():
    raise ValueError('Os identificadores originais não são inteiros válidos.')
brutas['source_id'] = id_numerico.astype('int64')
assert brutas['source_id'].is_unique, 'O identificador de origem está repetido.'

print('Linhas e colunas:', brutas.shape)
display(brutas.head())
diagnostico = pd.DataFrame({
    'tipo': brutas.dtypes.astype(str),
    'nulos': brutas.isna().sum(),
    'vazios_ou_espacos': [brutas[c].astype('string').str.strip().eq('').sum()
                         for c in brutas.columns]
})
display(diagnostico)


Linhas e colunas: (1407328, 7)


,source_id,headline,url,publisher,date,stock,source_row
0,0,Stocks That Hit 52-Week Highs On Friday,https://www.benzinga.com/news/20/06/16190091/s...,Benzinga Insights,2020-06-05 10:30:54-04:00,A,1
1,1,Stocks That Hit 52-Week Highs On Wednesday,https://www.benzinga.com/news/20/06/16170189/s...,Benzinga Insights,2020-06-03 10:45:20-04:00,A,2
2,2,71 Biggest Movers From Friday,https://www.benzinga.com/news/20/05/16103463/7...,Lisa Levin,2020-05-26 04:30:07-04:00,A,3
3,3,46 Stocks Moving In Friday's Mid-Day Session,https://www.benzinga.com/news/20/05/16095921/4...,Lisa Levin,2020-05-22 12:45:06-04:00,A,4
4,4,B of A Securities Maintains Neutral on Agilent...,https://www.benzinga.com/news/20/05/16095304/b...,Vick Meyer,2020-05-22 11:38:59-04:00,A,5


,tipo,nulos,vazios_ou_espacos
source_id,int64,0,0
headline,string,0,0
url,string,0,0
publisher,string,0,0
date,string,0,0
stock,string,0,0
source_row,int64,0,0


### 3.2 Selecionar empresas, validar datas e aplicar o período

Padronizamos os tickers e preservamos os valores originais. `news_date` usa o dia escrito na fonte; a conversão em UTC serve apenas para validar o timestamp, sem deslocar a data analítica. Registramos separadamente empresas fora do escopo, datas inválidas e registros fora de 2017–2019. Cada exclusão recebe um motivo principal. Reinicie o kernel para refazer a limpeza desde o começo.

In [3]:
brutas['stock_original'] = brutas['stock']
brutas['stock'] = brutas['stock'].str.strip().str.upper()
empresas_ok = brutas['stock'].isin(TICKERS)
fora_empresas = brutas.loc[~empresas_ok,
                           ['source_id', 'source_row', 'stock', 'date']].copy()
fora_empresas['motivo'] = 'fora_das_empresas'
fora_empresas.to_csv(TRATADOS / 'fora_das_empresas.csv.gz', index=False,
                     compression='gzip')
noticias = brutas.loc[empresas_ok].copy()
total_original = len(brutas)
total_empresas = len(noticias)
del brutas

noticias['date_original'] = noticias['date']
texto_data = noticias['date_original'].str.strip()
timestamp_valido = pd.to_datetime(texto_data, format='mixed', errors='coerce', utc=True).notna()
dia_texto = texto_data.str.extract(r'^(\d{4}-\d{2}-\d{2})(?:$|[ T])', expand=False)
noticias['news_date'] = pd.to_datetime(dia_texto, format='%Y-%m-%d', errors='coerce')
data_invalida = ~timestamp_valido | noticias['news_date'].isna()

exclusoes = []
def excluir(mask, motivo):
    global noticias
    retiradas = noticias.loc[mask].copy()
    retiradas['motivo'] = motivo
    retiradas['source_id_mantido'] = pd.NA
    exclusoes.append(retiradas)
    noticias = noticias.loc[~mask].copy()
    print(motivo, ':', len(retiradas))

excluir(data_invalida, 'data_invalida')
excluir(~noticias['news_date'].between(INICIO, FIM), 'fora_do_periodo')
total_recorte = len(noticias)
print('Original:', total_original, '| cinco empresas:', total_empresas,
      '| empresas e anos:', total_recorte)
display(noticias.groupby('stock').size().reindex(TICKERS).rename('noticias').to_frame())
display(noticias[['date_original', 'news_date']].head())


data_invalida : 0
fora_do_periodo : 7149
Original: 1407328 | cinco empresas: 13705 | empresas e anos: 6556


,noticias
stock,
NVDA,1649
NFLX,2344
BAC,871
JNJ,1047
KO,645


,date_original,news_date
133368,2019-12-27 00:00:00,2019-12-27
133369,2019-12-24 00:00:00,2019-12-24
133370,2019-12-22 00:00:00,2019-12-22
133371,2019-12-21 00:00:00,2019-12-21
133372,2019-12-16 00:00:00,2019-12-16


### 3.3 Normalizar manchetes e sinalizar ausências

Preservamos `headline_original` e normalizamos somente espaços. Pontuação, números e negações permanecem, pois afetam o significado para o FinBERT. Manchetes vazias são excluídas com registro; URLs e autores ausentes recebem sinalizadores. Não aplicamos aqui filtros de retrospectivas ou quantidade de notícias.

In [4]:
noticias['headline_original'] = noticias['headline']
noticias['headline'] = noticias['headline'].str.replace(r'\s+', ' ', regex=True).str.strip()
noticias['url_ausente'] = noticias['url'].str.strip().eq('')
noticias['publisher_ausente'] = noticias['publisher'].str.strip().eq('')
excluir(noticias['headline'].isna() | noticias['headline'].eq(''), 'manchete_vazia')
print('Manchetes com ajuste de espaços:',
      noticias['headline'].ne(noticias['headline_original']).sum())
display(noticias[['headline_original', 'headline', 'url_ausente', 'publisher_ausente']].head())


manchete_vazia : 0
Manchetes com ajuste de espaços: 0


,headline_original,headline,url_ausente,publisher_ausente
133368,Stocks That Hit 52-Week Highs On Friday,Stocks That Hit 52-Week Highs On Friday,False,False
133369,Achieving Real-Time Gross Settlement With On C...,Achieving Real-Time Gross Settlement With On C...,False,False
133370,How The Dow Jones Industrial Average Changed O...,How The Dow Jones Industrial Average Changed O...,False,False
133371,"Barron's Picks And Pans: Alibaba, Biogen, Chin...","Barron's Picks And Pans: Alibaba, Biogen, Chin...",False,False
133372,Stocks That Hit 52-Week Highs On Monday,Stocks That Hit 52-Week Highs On Monday,False,False


### 3.4 Remover duplicatas com rastreabilidade

A chave é `stock + news_date + headline`. Ordenamos por ID e mantemos a primeira ocorrência, registrando no excluído o ID preservado. URLs e IDs diferentes não tornam duas associações distintas para esta análise. A asserção final confere que não restaram repetições. Na referência, quatro ocorrências excedentes são retiradas.

In [5]:
CHAVE = ['stock', 'news_date', 'headline']

# Ordenar para manter o menor identificador original.
noticias = noticias.sort_values('source_id').copy()

# Mostrar todos os registros envolvidos nas repetições.
pares = noticias.loc[
    noticias.duplicated(CHAVE, keep=False)
].copy()

display(
    pares[['source_id', 'stock', 'news_date', 'headline', 'url']]
)

# Identificar qual registro será mantido em cada grupo.
id_mantido = noticias.groupby(CHAVE)['source_id'].transform('min')

# Marcar apenas as ocorrências excedentes.
duplicada = noticias.duplicated(CHAVE, keep='first')

# Documentar as exclusões antes de retirar as linhas.
retiradas = noticias.loc[duplicada].copy()
retiradas['motivo'] = 'duplicata_empresa_data_manchete'
retiradas['source_id_mantido'] = id_mantido.loc[duplicada]

if not retiradas.empty:
    exclusoes.append(retiradas)

# Manter uma ocorrência por empresa, data e manchete.
noticias = noticias.loc[~duplicada].copy()

assert not noticias.duplicated(CHAVE).any()

print('Excedentes retirados:', len(retiradas))
print('Notícias mantidas:', len(noticias))

Excedentes retirados: 4
Notícias mantidas: 6552


,source_id,stock,news_date,headline,url
704358,707934,JNJ,2017-06-23,Healthcare ETF (IYH) Hits New 52-Week High,https://www.benzinga.com/17/06/9646888/healthc...
704359,707935,JNJ,2017-06-23,Healthcare ETF (IYH) Hits New 52-Week High,https://www.benzinga.com/17/06/9646889/healthc...
731381,735140,KO,2019-02-27,Coca-Cola Option Alert: Jun 21 $42 Puts Sweep ...,https://www.benzinga.com/markets/options/19/02...
731382,735141,KO,2019-02-27,Coca-Cola Option Alert: Jun 21 $42 Puts Sweep ...,https://www.benzinga.com/markets/options/19/02...
924961,929729,NVDA,2019-03-19,45 Stocks Moving In Tuesday's Mid-Day Session,https://www.benzinga.com/news/19/03/13388164/4...
924962,929730,NVDA,2019-03-19,45 Stocks Moving In Tuesday's Mid-Day Session,https://www.benzinga.com/news/19/03/13388052/4...
925948,930716,NVDA,2017-06-14,Intel (INTC) Unveils Games & Announces Partner...,https://www.benzinga.com/17/06/9608349/intel-i...
925949,930717,NVDA,2017-06-14,Intel (INTC) Unveils Games & Announces Partner...,https://www.benzinga.com/17/06/9608350/intel-i...


### 3.5 Marcar manchetes compartilhadas

Contamos empresas distintas por data/manchete e sinalizamos grupos com mais de uma. Essas associações são preservadas: compartilhar um texto entre empresas difere de duplicá-lo dentro de uma empresa/data. A referência tem 301 grupos compartilhados. O vínculo fornecido pela base não garante que o título cite cada ticker associado.

In [6]:
grupo = noticias.groupby(['news_date', 'headline'])['stock']
noticias['n_empresas_manchete'] = grupo.transform('nunique')
noticias['compartilhada_entre_empresas'] = noticias['n_empresas_manchete'].gt(1)
compartilhadas = noticias.loc[noticias['compartilhada_entre_empresas']].copy()
print('Combinações data/manchete compartilhadas:',
      compartilhadas[['news_date', 'headline']].drop_duplicates().shape[0])
display(compartilhadas[['stock', 'news_date', 'headline', 'n_empresas_manchete']].head(10))
compartilhadas.to_csv(TRATADOS / 'manchetes_compartilhadas.csv', index=False,
                      date_format='%Y-%m-%d')


Combinações data/manchete compartilhadas: 301


,stock,news_date,headline,n_empresas_manchete
133370,BAC,2019-12-22,How The Dow Jones Industrial Average Changed O...,3
133371,BAC,2019-12-21,"Barron's Picks And Pans: Alibaba, Biogen, Chin...",2
133375,BAC,2019-12-14,"Barron's Picks And Pans: Charles Schwab, Disne...",2
133376,BAC,2019-12-13,Stocks That Hit 52-Week Highs On Friday,2
133377,BAC,2019-12-12,Stocks That Hit 52-Week Highs On Thursday,2
133381,BAC,2019-12-11,There's A New Trading Tool That Allows Traders...,2
133382,BAC,2019-12-10,Investor Movement Index: November 2019,2
133399,BAC,2019-11-09,"Barron's Picks And Pans: Honeywell, Oshkosh, U...",2
133404,BAC,2019-11-05,October's IMX Reveals Traders Making The Most ...,2
133406,BAC,2019-11-05,"Uber Shares Hit The Curb After Earnings, But T...",2


### 3.6 Auditar, exportar e conferir a releitura

Reconciliamos entradas, exclusões e notícias mantidas; verificamos datas, IDs e duplicatas. Na referência: **1.407.328 = 1.393.623 fora das empresas + 7.149 fora do período + 4 duplicatas + 6.552 mantidas**. Salvamos os CSVs, relemos notícias e conferimos a preservação do original. As contagens intermediárias do resumo não devem ser somadas. Se uma asserção falhar, investigue a etapa anterior antes de continuar.

In [7]:
CHAVE = ['stock', 'news_date', 'headline']
LIMPEZA_NOTICIAS_VALIDADA = False
audit = pd.concat(exclusoes, ignore_index=True)
audit['source_id_mantido'] = pd.to_numeric(audit['source_id_mantido'], errors='coerce').astype('Int64')
noticias = noticias.sort_values(['stock', 'news_date', 'source_id']).reset_index(drop=True)
noticias = noticias.drop(columns=['date'], errors='ignore')
assert noticias['news_date'].between(INICIO, FIM).all()
assert noticias['headline'].notna().all() and noticias['headline'].str.len().gt(0).all()
assert noticias['source_id'].is_unique
assert not noticias.duplicated(CHAVE).any()
assert noticias['source_id'].isin(audit['source_id']).sum() == 0
assert total_original == len(fora_empresas) + len(audit) + len(noticias)

motivos = ['data_invalida', 'fora_do_periodo', 'manchete_vazia',
           'duplicata_empresa_data_manchete']
contagens = audit['motivo'].value_counts().reindex(motivos, fill_value=0)
resumo_noticias = pd.DataFrame({
    'etapa': ['original', 'fora_das_empresas', 'cinco_empresas', *motivos,
              'recorte_antes_da_limpeza', 'noticias_limpas'],
    'quantidade': [total_original, len(fora_empresas), total_empresas,
                   *contagens.tolist(), total_recorte, len(noticias)]
})
display(resumo_noticias)
resumo_noticias.to_csv(DOCS / 'resumo_limpeza_noticias.csv', index=False)
audit.to_csv(TRATADOS / 'exclusoes_cinco_empresas.csv', index=False,
             date_format='%Y-%m-%d')
noticias.to_csv(TRATADOS / 'noticias_limpas.csv', index=False, date_format='%Y-%m-%d')
reabertas = pd.read_csv(TRATADOS / 'noticias_limpas.csv', parse_dates=['news_date'],
                        keep_default_na=False)
assert len(reabertas) == len(noticias)
assert reabertas['source_id'].tolist() == noticias['source_id'].tolist()
assert reabertas['headline'].tolist() == noticias['headline'].tolist()
assert reabertas['news_date'].equals(noticias['news_date'])
assert sha256(ARQUIVO_NOTICIAS) == HASH_ORIGINAL
print('Notícias exportadas e relidas; original preservado.')

LIMPEZA_NOTICIAS_VALIDADA = True


Notícias exportadas e relidas; original preservado.


,etapa,quantidade
0,original,1407328
1,fora_das_empresas,1393623
2,cinco_empresas,13705
3,data_invalida,0
4,fora_do_periodo,7149
5,manchete_vazia,0
6,duplicata_empresa_data_manchete,4
7,recorte_antes_da_limpeza,6556
8,noticias_limpas,6552


### 3.7 Conferir a cobertura mensal e anual

Contamos associações por empresa/mês e empresa/ano. Reindexar todos os 36 meses mostra zeros de cobertura; zero aqui significa ausência de registros no corpus, sem atribuir neutralidade. As tabelas são exportadas para conferência. O gráfico e a interpretação ficam na pergunta 1 do [artigo](00_artigo.ipynb).

In [8]:
meses = pd.period_range(INICIO, FIM, freq='M')
cobertura = (noticias.assign(mes=noticias['news_date'].dt.to_period('M'))
             .groupby(['stock', 'mes']).size().unstack(fill_value=0)
             .reindex(index=TICKERS, columns=meses, fill_value=0))
cobertura.columns = cobertura.columns.astype(str)
cobertura.to_csv(TRATADOS / 'cobertura_noticias_mensal.csv', index_label='stock')
por_ano = noticias.groupby(['stock', noticias['news_date'].dt.year]).size().unstack(fill_value=0)
por_ano.to_csv(DOCS / 'cobertura_noticias_anual.csv', index_label='stock')
display(por_ano.reindex(TICKERS))

news_date,2017,2018,2019
stock,,,
NVDA,614,526,507
NFLX,759,784,801
BAC,348,220,303
JNJ,300,314,432
KO,238,195,211


### 3.8 Obter cotações com margem temporal

Reutilizamos o snapshot quando seus parâmetros e hash conferem. Caso não exista, coletamos cada empresa com `auto_adjust=False`, preservando `Adj Close`, volume e eventos. O intervalo vai de 15/12/2016 a 10/01/2020; `end` é exclusivo. Uma falha de coleta interrompe a etapa. Dados ajustados podem mudar na fonte, por isso o CSV e seus metadados são conservados juntos.

In [9]:
import yfinance as yf
yf.set_tz_cache_location(str(RAIZ / 'tmp' / 'yfinance'))
ARQUIVO_PRECOS = ORIGINAIS / 'cotacoes_yahoo_margem.csv'
METADADOS_PRECOS = ORIGINAIS / 'cotacoes_yahoo_margem.json'
parametros = dict(start=COLETA_INICIO, end=COLETA_FIM_EXCLUSIVO, interval='1d',
                  auto_adjust=False, back_adjust=False, repair=False, keepna=True,
                  actions=True, ignore_tz=True, threads=False, progress=False,
                  multi_level_index=False, timeout=30)
if ARQUIVO_PRECOS.exists():
    if not METADADOS_PRECOS.exists():
        raise ValueError('Snapshot de preços sem metadados: documente a origem antes de usar.')
    coleta_precos = json.loads(METADADOS_PRECOS.read_text(encoding='utf-8'))
    assert coleta_precos['tickers'] == TICKERS
    assert coleta_precos['parametros'] == parametros, 'Parâmetros mudaram: reveja o snapshot.'
    assert coleta_precos['sha256'] == sha256(ARQUIVO_PRECOS)
    precos_brutos = pd.read_csv(ARQUIVO_PRECOS, keep_default_na=False)
    print('Snapshot existente reutilizado.')
else:
    tabelas = []
    for ticker in TICKERS:
        tabela = yf.download(ticker, **parametros)
        if tabela is None or tabela.empty:
            raise RuntimeError(f'A coleta de {ticker} falhou. Corrija e execute esta célula novamente.')
        if isinstance(tabela.columns, pd.MultiIndex):
            raise ValueError(f'Colunas inesperadas em {ticker}: confira a versão do yfinance.')
        if not {'Adj Close', 'Volume'}.issubset(tabela.columns):
            raise ValueError(f'Fechamento ajustado/volume ausente em {ticker}.')
        tabela = tabela.rename_axis('trade_date').reset_index()
        tabela['stock'] = ticker
        tabelas.append(tabela)
    precos_brutos = pd.concat(tabelas, ignore_index=True)
    precos_brutos.to_csv(ARQUIVO_PRECOS, index=False, date_format='%Y-%m-%d')
    coleta_precos = {'fonte': 'Yahoo Finance via yfinance', 'tickers': TICKERS,
                     'parametros': parametros, 'yfinance': version('yfinance'),
                     'coletado_em_utc': datetime.now(timezone.utc).isoformat(),
                     'sha256': sha256(ARQUIVO_PRECOS)}
    METADADOS_PRECOS.write_text(json.dumps(coleta_precos, indent=2, ensure_ascii=False), encoding='utf-8')
display(precos_brutos.head())
print('Coleta registrada em:', coleta_precos['coletado_em_utc'])


Snapshot existente reutilizado.
Coleta registrada em: 2026-10-09T21:42:24.945771+00:00


,trade_date,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume,stock
0,2016-12-15,2.425262,2.46775,0.0,2.49850,2.41500,2.42250,0.0,746548000,NVDA
1,2016-12-16,2.467031,2.51025,0.0,2.51750,2.44325,2.49575,0.0,1089528000,NVDA
2,2016-12-19,2.497005,2.54075,0.0,2.56125,2.47500,2.49250,0.0,744656000,NVDA
3,2016-12-20,2.583982,2.62925,0.0,2.65325,2.60300,2.61450,0.0,848056000,NVDA
4,2016-12-21,2.600199,2.64575,0.0,2.64875,2.59275,2.64100,0.0,576136000,NVDA


### 3.9 Validar cotações e cobertura

Convertemos datas e números, exigimos fechamento ajustado finito e positivo e volume válido. Duplicatas com valores divergentes interrompem a etapa; repetições idênticas são registradas. Comparamos as datas entre empresas, conferimos 754 pregões por empresa no recorte e as margens, sem interpolar preços. A comparação externa com XNYS ocorre no notebook 03, pois uma data poderia faltar em todas as séries. Execute as duas células abaixo em ordem.

In [10]:
precos = precos_brutos.rename(columns={'Adj Close': 'adj_close', 'Volume': 'volume'}).copy()
precos['price_source_row'] = np.arange(1, len(precos) + 1)
precos['trade_date_original'] = precos['trade_date'].astype('string')
precos['trade_date'] = pd.to_datetime(precos['trade_date_original'].str[:10],
                                      format='%Y-%m-%d', errors='coerce')
precos['stock'] = precos['stock'].astype('string').str.strip().str.upper()
precos['adj_close'] = pd.to_numeric(precos['adj_close'], errors='coerce')
precos['volume'] = pd.to_numeric(precos['volume'], errors='coerce')
precos['volume_invalido'] = (~np.isfinite(precos['volume']) | precos['volume'].lt(0)
                            | precos['volume'].mod(1).ne(0))
precos['volume_zero'] = precos['volume'].eq(0)
precos['motivo_preco'] = np.select([
    ~precos['stock'].isin(TICKERS),
    precos['trade_date'].isna(),
    ~precos['trade_date'].between(pd.Timestamp(COLETA_INICIO),
                                 pd.Timestamp(COLETA_FIM_EXCLUSIVO) - pd.Timedelta(days=1)),
    ~np.isfinite(precos['adj_close']) | precos['adj_close'].le(0)
], ['empresa_invalida', 'data_invalida', 'fora_da_margem', 'fechamento_ajustado_invalido'], default='')
problemas_precos = precos.loc[precos['motivo_preco'].ne('')].copy()
problemas_precos.to_csv(TRATADOS / 'problemas_precos.csv', index=False, date_format='%Y-%m-%d')
precos = precos.loc[precos['motivo_preco'].eq('')].copy()

chave_preco = ['stock', 'trade_date']
repetidos = precos.loc[precos.duplicated(chave_preco, keep=False)].copy()
if not repetidos.empty:
    campos_valor = [c for c in precos_brutos.columns if c not in ['stock', 'trade_date']]
    campos_valor = [{'Adj Close': 'adj_close', 'Volume': 'volume'}.get(c, c) for c in campos_valor]
    variacoes = repetidos.groupby(chave_preco)[campos_valor].nunique(dropna=False)
    if variacoes.gt(1).any().any():
        repetidos.to_csv(TRATADOS / 'duplicatas_precos_divergentes.csv', index=False)
        raise ValueError('Empresa/pregão com valores divergentes. Investigue o arquivo de duplicatas.')
duplicatas_precos = precos.loc[precos.duplicated(chave_preco, keep='first')].copy()
duplicatas_precos.to_csv(TRATADOS / 'duplicatas_precos.csv', index=False, date_format='%Y-%m-%d')
precos = precos.drop_duplicates(chave_preco).sort_values(chave_preco).reset_index(drop=True)
precos['no_recorte'] = precos['trade_date'].between(INICIO, FIM)
print('Preços inválidos:', len(problemas_precos), '| duplicatas retiradas:', len(duplicatas_precos))
print('Volumes a revisar:', precos['volume_invalido'].sum(), '| volumes zero:', precos['volume_zero'].sum())
display(precos.groupby('stock').agg(primeiro=('trade_date', 'min'), ultimo=('trade_date', 'max'),
                                   registros=('trade_date', 'size')))


Preços inválidos: 0 | duplicatas retiradas: 0
Volumes a revisar: 0 | volumes zero: 0


,primeiro,ultimo,registros
stock,,,
BAC,2016-12-15,2020-01-10,772
JNJ,2016-12-15,2020-01-10,772
KO,2016-12-15,2020-01-10,772
NFLX,2016-12-15,2020-01-10,772
NVDA,2016-12-15,2020-01-10,772


In [11]:
calendario = pd.DatetimeIndex(sorted(precos['trade_date'].unique()))
lacunas = []
for ticker in TICKERS:
    observados = pd.DatetimeIndex(precos.loc[precos['stock'].eq(ticker), 'trade_date'])
    for dia in calendario.difference(observados):
        lacunas.append({'stock': ticker, 'trade_date': dia, 'motivo': 'ausente_em_relacao_as_outras_empresas'})
lacunas = pd.DataFrame(lacunas, columns=['stock', 'trade_date', 'motivo'])
lacunas.to_csv(TRATADOS / 'lacunas_precos.csv', index=False, date_format='%Y-%m-%d')
resumo_precos = precos.loc[precos['no_recorte']].groupby('stock').agg(
    pregoes=('trade_date', 'size'), primeiro=('trade_date', 'min'), ultimo=('trade_date', 'max'))
resumo_precos = resumo_precos.reindex(TICKERS)
resumo_precos['referencia_proposta'] = 754
resumo_precos['diferenca_referencia'] = resumo_precos['pregoes'] - 754
display(resumo_precos)
display(lacunas.head(10))
resumo_precos.to_csv(DOCS / 'cobertura_precos.csv', index_label='stock', date_format='%Y-%m-%d')

assert len(problemas_precos) == 0, 'Há preços inválidos; investigue/recolete antes de liberar.'
assert len(lacunas) == 0, 'Há lacunas entre empresas; investigue/recolete antes de liberar.'
assert resumo_precos['pregoes'].notna().all(), 'Falta uma empresa.'
assert resumo_precos['diferenca_referencia'].eq(0).all(), 'Contagem difere da proposta: investigue antes de liberar.'
assert not precos['volume_invalido'].any(), 'Há volumes inválidos: investigue antes de liberar.'
for ticker in TICKERS:
    dias = precos.loc[precos['stock'].eq(ticker), 'trade_date']
    assert dias.lt(INICIO).any(), f'Falta margem anterior em {ticker}.'
    assert dias.gt(FIM).any(), f'Falta margem posterior em {ticker}.'
assert not precos.duplicated(chave_preco).any()
precos = precos.drop(columns='motivo_preco')
precos.to_csv(TRATADOS / 'cotacoes_limpas_com_margem.csv', index=False, date_format='%Y-%m-%d')
precos_reabertos = pd.read_csv(TRATADOS / 'cotacoes_limpas_com_margem.csv',
                              parse_dates=['trade_date'], keep_default_na=False)
assert len(precos_reabertos) == len(precos)
assert precos_reabertos['trade_date'].equals(precos['trade_date'])
np.testing.assert_allclose(precos_reabertos['adj_close'], precos['adj_close'])
print('Cotações conferidas, exportadas e relidas.')


Cotações conferidas, exportadas e relidas.


,pregoes,primeiro,ultimo,referencia_proposta,diferenca_referencia
stock,,,,,
NVDA,754,2017-01-03,2019-12-31,754,0
NFLX,754,2017-01-03,2019-12-31,754,0
BAC,754,2017-01-03,2019-12-31,754,0
JNJ,754,2017-01-03,2019-12-31,754,0
KO,754,2017-01-03,2019-12-31,754,0


,stock,trade_date,motivo


### 3.10 Registrar a preparação

O dicionário descreve as colunas; o manifesto registra versões, parâmetros, contagens e hashes; o resumo documenta as decisões. Esta célula depende das auditorias anteriores concluídas. Os registros descrevem esta etapa de preparação, não o andamento de todo o projeto.

In [12]:
if not globals().get('LIMPEZA_NOTICIAS_VALIDADA', False):
    raise RuntimeError('Execute a célula 3.6 de auditoria até o fim, sem erros, antes de documentar a conclusão.')
descricoes = {
 'source_id': 'Identificador inteiro preservado do índice exportado no CSV original.',
 'source_row': 'Número da linha lógica no CSV original, começando em 1 após o cabeçalho.',
 'stock': 'Ticker normalizado da empresa associada no dataset.',
 'stock_original': 'Ticker exatamente como lido no original.',
 'headline': 'Manchete para FinBERT; somente espaços normalizados.',
 'headline_original': 'Texto original preservado.',
 'url': 'URL original; ausência isolada não exclui a manchete.',
 'publisher': 'Autor ou identificação da fonte conforme o dataset; significado pode variar.',
 'date_original': 'Data/hora original em texto; não se presume horário confiável.',
 'news_date': 'Dia registrado na notícia, sem deslocamento por conversão de fuso.',
 'url_ausente': 'Sinalizador de URL vazia ou composta por espaços.',
 'publisher_ausente': 'Sinalizador de publisher vazio ou composto por espaços.',
 'n_empresas_manchete': 'Quantidade de tickers distintos para a mesma data/manchete na base limpa.',
 'compartilhada_entre_empresas': 'Verdadeiro se a mesma data/manchete estiver associada a várias empresas.',
 'trade_date': 'Data do pregão observada na coleta diária.',
 'adj_close': 'Fechamento ajustado da fonte, em dólares; preservado sem ajustes próprios.',
 'volume': 'Volume diário informado na coleta; não é preenchido.',
 'price_source_row': 'Linha lógica do snapshot de cotações.',
 'trade_date_original': 'Data do snapshot preservada em texto.',
 'volume_invalido': 'Sinalizador de volume ausente, não finito, negativo ou não inteiro.',
 'volume_zero': 'Sinalizador de volume zero preservado para revisão.',
 'no_recorte': 'Verdadeiro entre 2017-01-01 e 2019-12-31; margens ficam na tabela.',
 'Open': 'Abertura não ajustada retornada pelo Yahoo, em dólares.',
 'High': 'Máxima não ajustada retornada pelo Yahoo, em dólares.',
 'Low': 'Mínima não ajustada retornada pelo Yahoo, em dólares.',
 'Close': 'Fechamento não ajustado retornado pelo Yahoo, em dólares.',
 'Dividends': 'Dividendo por ação informado na data do evento.',
 'Stock Splits': 'Fator de desdobramento informado na data do evento.'
}
linhas = []
for nome, tabela in [('noticias_limpas', noticias), ('cotacoes_limpas_com_margem', precos)]:
    for coluna in tabela.columns:
        if coluna not in descricoes:
            raise ValueError(f'Descreva a nova coluna antes de liberar: {coluna}')
        linhas.append({'tabela': nome, 'coluna': coluna, 'tipo_no_notebook': str(tabela[coluna].dtype),
                       'descricao': descricoes[coluna]})
pd.DataFrame(linhas).to_csv(DOCS / 'dicionario_dados.csv', index=False)
manifesto = {
 'etapa': 'limpeza de notícias e conferência de cotações',
 'gerado_em_utc': datetime.now(timezone.utc).isoformat(), 'tickers': TICKERS,
 'inicio': str(INICIO.date()), 'fim': str(FIM.date()), 'versoes': VERSOES,
 'ambiente_execucao': {'MKL_THREADING_LAYER': os.environ.get('MKL_THREADING_LAYER')},
 'noticias_fonte': 'https://www.kaggle.com/datasets/miguelaenlle/massive-stock-news-analysis-db-for-nlpbacktests',
 'noticias_sha256_original': HASH_ORIGINAL, 'coleta_precos': coleta_precos,
 'original_noticias': total_original, 'recorte_antes_limpeza': total_recorte,
 'noticias_limpas': len(noticias), 'exclusoes_nas_cinco_empresas': contagens.to_dict(),
 'hashes_resultados': {p.name: sha256(p) for p in
                      [TRATADOS / 'noticias_limpas.csv', TRATADOS / 'cotacoes_limpas_com_margem.csv']},
 'unidade_noticias': 'empresa, dia registrado, manchete normalizada',
 'unidade_precos': 'empresa, pregão observado',
 'continuidade': ['02_sentimento_finbert.ipynb', '03_integracao_retornos.ipynb']
}
(DOCS / 'manifesto_limpeza.json').write_text(json.dumps(manifesto, indent=2, ensure_ascii=False), encoding='utf-8')
texto_metodo = f"""# Registro da limpeza

O recorte contém {total_recorte:,} associações de notícias antes da limpeza e {len(noticias):,} depois.
Filtramos NVDA, NFLX, BAC, JNJ e KO entre 2017 e 2019, preservando identificadores e campos originais.
Validamos datas, excluímos manchetes vazias e removemos repetições por empresa/data/manchete, mantendo o menor source_id.
Os registros compartilhados entre empresas foram marcados e mantidos. As URLs dos pares repetidos foram preservadas na auditoria.
Não excluímos manchetes retrospectivas nem dias com menos de três notícias da base principal.
As cotações têm margem entre {COLETA_INICIO} e {COLETA_FIM_EXCLUSIVO} (fim exclusivo), fechamento ajustado e volume.
Verificamos duplicatas, preços, volumes, cobertura e margens, sem preencher preços ausentes.
O calendário foi conferido pela comparação entre as empresas; falhas comuns a todas as séries ainda exigem atenção.
Os resultados descrevem associações; esta etapa não executa FinBERT nem identifica causalidade.

Consulte resumo_limpeza_noticias.csv, cobertura_precos.csv, dicionario_dados.csv e manifesto_limpeza.json nesta pasta.
"""
(DOCS / 'registro_limpeza.md').write_text(texto_metodo, encoding='utf-8')
assert sha256(ARQUIVO_NOTICIAS) == HASH_ORIGINAL
assert sha256(ARQUIVO_PRECOS) == coleta_precos['sha256']
print('ETAPA CONCLUÍDA: notícias prontas para FinBERT e cotações prontas para a integração posterior.')
print('Arquivos: data/processed/noticias_limpas.csv e data/processed/cotacoes_limpas_com_margem.csv')


ETAPA CONCLUÍDA: notícias prontas para FinBERT e cotações prontas para a integração posterior.
Arquivos: data/processed/noticias_limpas.csv e data/processed/cotacoes_limpas_com_margem.csv


## 4. Continuidade

Use as notícias limpas no [notebook 02](02_sentimento_finbert.ipynb), depois execute a [integração](03_integracao_retornos.ipynb) e o [artigo](00_artigo.ipynb). O README apresenta a ordem completa de reprodução.